# Transformer

Train models on the generated tokens, snapshot them, probe the snapshots, and
compare one sequence against ground truth.

This notebook builds its own process from the same free-text controls as
`simulator.ipynb`, so it stands alone - but the two agree exactly when the
settings match, because both call `make_process`.

**A saved checkpoint pins its own vocabulary and context.** The committed runs are
`181` bins and `160` positions. Change `bins per channel` or the tick structure and
those checkpoints no longer apply; the notebook says so rather than silently
producing nonsense. Train a fresh model in the Train tab for any other setting.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import ipywidgets as W
import numpy as np
from IPython.display import display

from models.explore import find_checkpoints, fit_readout, load_model, plot_sequence
from models.lab import Arch, probe_checkpoints, run_training, tunable_fields
from models.labplots import plot_probe_table, plot_training
from models.train import pick_device
from physics.messk_configs import CHAIN, DRIVER, MESSK_CONFIGS, SYSTEMS, make_process

DEVICE = pick_device()
CHECKPOINT_DIR = Path("../experiments/outputs-messk")
SESSION_DIR = Path("../experiments/outputs-lab")
SESSION_DIR.mkdir(parents=True, exist_ok=True)
STATE = {"proc": None, "batch": None, "report": None, "rows": None, "found": {}}


def num(label, value, width="230px"):
    cls = W.IntText if isinstance(value, (int, np.integer)) and not isinstance(value, bool) else W.FloatText
    return cls(value=value, description=label, style={"description_width": "125px"},
               layout=W.Layout(width=width))


def all_checkpoints():
    found = find_checkpoints(CHECKPOINT_DIR) | find_checkpoints(SESSION_DIR)
    STATE["found"] = found
    return found

print(f"device {DEVICE} | {len(all_checkpoints())} checkpoints visible")

## Data

The process these models are trained on and probed against. Same controls as the
simulator notebook; see that one for what the bin syntax means.

In [ ]:
sys_w     = W.Dropdown(options=sorted(MESSK_CONFIGS), value="pendulum_mess4",
                       description="system", style={"description_width": "125px"})
delta_v_w = num("delta_v", SYSTEMS["pendulum"]["delta_v"])
dt_w      = num("dt", SYSTEMS["pendulum"]["dt"])
n_steps_w = num("physics steps/tick", DRIVER["n_steps"])
m_w       = num("chain ticks (m)", DRIVER["m"])
bins_w    = W.Text(value="181", description="bins per channel",
                   style={"description_width": "125px"}, layout=W.Layout(width="300px"))
ranges_w  = W.Textarea(value="", description="channel ranges",
                       style={"description_width": "125px"},
                       layout=W.Layout(width="300px", height="52px"))
stay_w    = num("stay", CHAIN["stay"])
alpha_w   = num("alpha", CHAIN["alpha"])
nseq_w    = num("sequences", 256)
seed_w    = num("seed", 20260829)
data_b    = W.Button(description="build data", button_style="primary")
data_out  = W.Output()
physics_box = W.VBox([])

def _rebuild(*_):
    name = MESSK_CONFIGS[sys_w.value]
    system = SYSTEMS[name]["factory"]()
    physics_box.children = [num(f, v) for f, v in tunable_fields(system).items()]
    delta_v_w.value, dt_w.value = SYSTEMS[name]["delta_v"], SYSTEMS[name]["dt"]
    ranges = list(getattr(system, "obs_ranges", (system.obs_range,)))
    ranges_w.value = "\n".join(f"{lo:.6g},{hi:.6g}" for lo, hi in ranges)

sys_w.observe(_rebuild, names="value")
_rebuild()

def build_proc():
    overrides = {w.description: w.value for w in physics_box.children}
    rows = [r for r in ranges_w.value.strip().splitlines() if r.strip()]
    overrides["obs_range"] = tuple(float(x) for x in rows[0].split(","))
    parts = [int(p) for p in bins_w.value.replace("x", ",").split(",") if p.strip()]
    return make_process(
        sys_w.value, system=overrides, chain={"stay": stay_w.value, "alpha": alpha_w.value},
        delta_v=delta_v_w.value, dt=dt_w.value, n_steps=int(n_steps_w.value),
        m=int(m_w.value), obs_bins=parts[0] if len(parts) == 1 else tuple(parts),
    )

def _build(_=None):
    with data_out:
        data_out.clear_output(wait=True)
        proc = build_proc()
        batch = proc.sample_batch(np.random.default_rng(int(seed_w.value)), int(nseq_w.value))
        STATE["proc"], STATE["batch"] = proc, batch
        r = proc.bin_report(batch["observable"])
        print(f"vocabulary {'x'.join(map(str, r['obs_bins']))} = {proc.n_obs} | "
              f"context {proc.seq_len} | uniform loss {np.log(proc.n_obs):.3f} nats | "
              f"{r['used_bins']} cells occupied | clipped {r['clipped']:.2%}")

data_b.on_click(_build)
display(W.VBox([W.HBox([
    W.VBox([W.HTML("<b>physics</b>"), sys_w, delta_v_w, dt_w, physics_box]),
    W.VBox([W.HTML("<b>channel</b>"), n_steps_w, m_w, bins_w, ranges_w]),
    W.VBox([W.HTML("<b>chain and sampling</b>"), stay_w, alpha_w, nseq_w, seed_w, data_b]),
]), data_out]))
_build()

## Train

Snapshots land in `experiments/outputs-lab/` and feed the probe below. Fraction
`0.0` saves before the first step: that is the random-initialisation control, and
it is the number every later result has to beat.

In [ ]:
d_model_w = num("d_model", 128)
layers_w  = num("n_layers", 4)
heads_w   = num("n_heads", 1)
dmlp_w    = num("d_mlp (0 = 4x)", 0)
tokens_w  = num("total tokens", 2_000_000, "260px")
batch_w   = num("batch size", 128)
lr_w      = num("learning rate", 1e-3)
frac_w    = W.Text(value="0.0,0.1,0.25,0.5,1.0", description="snapshot at",
                   style={"description_width": "125px"})
runname_w = W.Text(value="lab", description="run name", style={"description_width": "125px"})
train_b   = W.Button(description="train", button_style="primary")
train_out = W.Output()

def _train(_=None):
    with train_out:
        train_out.clear_output(wait=True)
        if STATE["proc"] is None:
            print("build the data first"); return
        arch = Arch(d_model=int(d_model_w.value), n_layers=int(layers_w.value),
                    n_heads=int(heads_w.value), d_mlp=int(dmlp_w.value) or None)
        started = time.perf_counter()
        report = run_training(
            STATE["proc"], arch, SESSION_DIR, name=runname_w.value,
            total_tokens=int(tokens_w.value), batch_size=int(batch_w.value),
            learning_rate=lr_w.value,
            fractions=tuple(float(f) for f in frac_w.value.split(",")), device=DEVICE,
        )
        report["batch_size"] = int(batch_w.value)
        STATE["report"] = report
        all_checkpoints()
        print(f"{report['steps']} steps in {time.perf_counter() - started:.0f}s | "
              f"final eval {report['final']['eval_loss']:.3f} against uniform "
              f"{report['uniform_loss']:.3f} nats | "
              f"{len(report['checkpoint_paths'])} snapshots")
        display(plot_training(report))

train_b.on_click(_train)
display(W.VBox([W.HBox([
    W.VBox([W.HTML("<b>architecture</b>"), d_model_w, layers_w, heads_w, dmlp_w]),
    W.VBox([W.HTML("<b>optimisation</b>"), tokens_w, batch_w, lr_w]),
    W.VBox([W.HTML("<b>run</b>"), frac_w, runname_w, train_b]),
]), train_out]))

## Probe

**Read the two bars against each other.** The metric bar is high even at snapshot
`0`, because the physical state is close to a linear function of the recent tokens
and an untrained network's random projection preserves that. The belief bar at
snapshot `0` is the bar any later result has to clear.

In [ ]:
source_w  = W.RadioButtons(options=["this session", "saved files"], description="models",
                           style={"description_width": "125px"})
saved_w   = W.SelectMultiple(options=sorted(all_checkpoints()), rows=9, description="files",
                             style={"description_width": "125px"},
                             layout=W.Layout(width="560px"))
depths_w  = num("readout depths", 4)
ridge_w   = num("ridge alpha", 100.0)
nfit_w    = num("fit sequences", 3072)
whole_w   = W.Checkbox(value=True, description="whole-tick readout")
probe_b   = W.Button(description="probe", button_style="primary")
probe_out = W.Output()

def _probe(_=None):
    with probe_out:
        probe_out.clear_output(wait=True)
        if STATE["proc"] is None:
            print("build the data first"); return
        if source_w.value == "this session":
            if STATE["report"] is None:
                print("no run in this session"); return
            paths = STATE["report"]["checkpoint_paths"]
        else:
            paths = {n: str(STATE["found"][n]) for n in saved_w.value}
            if not paths:
                print("select at least one file"); return
        rows = probe_checkpoints(
            STATE["proc"], paths, device=DEVICE, n_fit=int(nfit_w.value),
            n_depths=int(depths_w.value), whole_tick=whole_w.value, alpha=ridge_w.value)
        STATE["rows"] = rows
        for r in rows:
            print(f"{str(r['label']):>44s}  belief {r['belief_r2']:+.3f}  "
                  f"metric {r['metric_r2']:+.3f}  loss {r['eval_loss']:.3f}  "
                  f"({r['rows_per_feature']:.1f} rows/feature)")
        display(plot_probe_table(rows))

probe_b.on_click(_probe)
display(W.VBox([W.HBox([
    W.VBox([W.HTML("<b>which models</b>"), source_w, saved_w]),
    W.VBox([W.HTML("<b>readout</b>"), depths_w, whole_w, ridge_w, nfit_w, probe_b]),
]), probe_out]))

## Truth against model

One sequence, one checkpoint: exact belief and metric against the linear readout,
plus the model's own next-token distribution, which has no probe between it and
the picture.

In [ ]:
ckpt_w    = W.Dropdown(options=sorted(all_checkpoints()) or ["<none>"], description="checkpoint",
                       style={"description_width": "125px"}, layout=W.Layout(width="520px"))
refresh_b = W.Button(description="refresh")
compare_w = num("sequence", 0)
show_b    = W.Button(description="show", button_style="primary")
show_out  = W.Output()

def _refresh(_=None):
    ckpt_w.options = sorted(all_checkpoints()) or ["<none>"]

def _show(_=None):
    with show_out:
        show_out.clear_output(wait=True)
        if STATE["proc"] is None:
            print("build the data first"); return
        proc, batch = STATE["proc"], STATE["batch"]
        model = load_model(STATE["found"][ckpt_w.value], device=DEVICE)
        if proc.seq_len > model.config.n_ctx or proc.n_obs > model.config.vocab_size:
            print(f"this checkpoint was trained with vocabulary {model.config.vocab_size} "
                  f"and context {model.config.n_ctx}; the current data is "
                  f"{proc.n_obs} x {proc.seq_len}. Match the settings, or train a new model.")
            return
        readout = fit_readout(model, proc, device=DEVICE, n_fit=int(nfit_w.value),
                              n_depths=min(int(depths_w.value), model.config.n_layers + 1),
                              whole_tick=whole_w.value, alpha=ridge_w.value)
        print(f"{model.config.n_layers} layers, d_model {model.config.d_model} | "
              f"belief {readout.r2['belief']:+.3f}  metric {readout.r2['metric']:+.3f} "
              f"({readout.rows_per_feature:.1f} rows/feature)")
        display(plot_sequence(proc, batch, int(compare_w.value) % len(batch["tokens"]),
                              model, readout, DEVICE))

refresh_b.on_click(_refresh)
show_b.on_click(_show)
display(W.VBox([W.HBox([ckpt_w, refresh_b, compare_w, show_b]), show_out]))

## Things worth trying

- **Probe snapshot `0` against the final one.** The metric bar barely moves; the
  belief bar is where training shows up. That gap is the result of the experiment.
- **Two models at one budget, `d_model` 64 and 256.** The belief bars separate,
  the metric bars do not.
- **Turn `whole-tick` off.** The readout drops to one position per tick and the
  metric collapses, which isolates the readout effect from the model entirely.
- **Train the double pendulum at `181x181`.** The vocabulary jumps to 32761, so the
  uniform loss goes from 5.20 to 10.40 nats and losses stop being comparable to
  any single-channel run - but both joints are now visible in the token.